<br><br><br><br>
<div><img style="float: left; padding-right: 100px; width: 350px" src="../static/arifa.png"></div>
<br>
<h2 align="center">ARIFA</h2>
<hr>
<h4 align="center">Zephania Reuben &amp; Philipp Ramjoué</h4>
<h4 align="center">April, 2026</h4>
<hr>
<h3 align="center">AI / ML FOR ZHSF SYSTEM OPTIMIZATION<br><hr>
FROM RAW DATA TO TRAINED MODEL — DEPLOYMENT EDITION v2
</h3>
<hr><br>

## Overview

End-to-end deployment-ready ML pipeline for detecting suspicious claims in the ZHSF system.
Two models are trained for **A/B testing**: XGBoost (primary) and Random Forest (challenger).

### Bug Fixes vs v1
| # | Bug | Root Cause | Fix |
|---|-----|-----------|-----|
| 1 | XGBoost threshold ≈ 0.09 (too low) | `scale_pos_weight=23` applied on top of SMOTE-balanced data — double penalty | Set `scale_pos_weight=1` when data is already balanced by SMOTE |
| 2 | `hospital_claim_count` lookup used `_provider_ref` | Copy-paste error in `prepare_sample` | Use `_hospital_ref` for hospital lookups |
| 3 | Reference stats computed on full dataset | Minor leakage: val/test medians influenced training features | Compute medians on train split only, then apply to val/test |
| 4 | `rolling_30d_provider` is a cumcount, not a 30-day window | Misleading name — actual 30-day window never computed | Implemented true 30-day rolling window |

### Pipeline Steps
| Step | Section | Maps to `src/` |
|------|---------|----------------|
| 1 | Configuration & Imports | `config.py` |
| 2 | Load Data | `data.py → load_raw_data()` |
| 3 | Clean Data | `data.py → clean_claims()` |
| 4 | Feature Engineering | `features.py → build_features()` |
| 5 | Encode Categoricals | `features.py → encode_categoricals()` |
| 6 | Train / Val / Test Split | `data.py → chronological_split()` |
| 7 | SMOTE Oversampling | `train.py → smote_balance()` |
| 8 | Train XGBoost | `train.py → train_xgboost()` |
| 9 | Train Random Forest | `train.py → train_random_forest()` |
| 10 | Threshold Selection | `evaluate.py → find_f1_threshold()` |
| 11 | Final Test Evaluation | `evaluate.py → full_report()` |
| 12 | Save Artifacts | `artifacts.py → save_all()` |
| 13 | Smoke Test (2 samples) | `predict.py → predict_claim()` |

> **Dataset:** ZHSF 2022–2023 · 2,000 claims · 18 facilities · ~4% anomaly rate

---
## Step 1 — Configuration & Imports
**Maps to:** `src/config.py`

All paths and constants are defined here so every downstream cell reads from a single source of truth.
When converting to scripts, this block becomes `config.py`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION  →  src/config.py
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = '../data/raw/'        # raw CSV files live here
MODELS_DIR = '../models/'          # trained model artifacts saved here
LOGS_DIR   = '../logs/'            # training logs

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

# ── Feature columns (single source of truth) ───────────────────────────────
# These exact names must be produced by build_features() and consumed by
# every model. Order matters — models are sensitive to column order.
FEATURE_COLS = [
    # Group 1: Temporal signals
    'claim_age_days',          # days from service to submission
    'submission_month',        # 1–12
    'submission_dayofweek',    # 0=Mon … 6=Sun
    'submission_quarter',      # 1–4
    'is_weekend_submission',   # 1 if submitted Sat/Sun
    'is_malaria_season',       # 1 if April, May, Oct, Nov
    'log_claim_amount',        # log1p(claimed_amount_tzs) — tames outliers
    # Group 2: Provider / hospital frequency
    'provider_claim_count',    # total claims by this provider in training data
    'hospital_claim_count',    # total claims at this hospital in training data
    'rolling_30d_provider',    # claims by provider in the 30 days before this claim
    # Group 3: Cost-deviation signals
    'icd10_median_cost',       # reference median cost for this diagnosis code
    'cost_deviation_pct',      # % above/below ICD-10 median
    'amount_vs_hospital_median',# ratio: claimed / hospital median
    # Group 4: Encoded categoricals
    'facility_type_enc',
    'plan_type_enc',
    'service_type_enc',
    'patient_gender_enc',
    'patient_district_enc',
]

# ── Categorical columns to encode ──────────────────────────────────────────
CATEGORICAL_COLS = {
    'facility_type'   : 'facility_type_enc',
    'plan_type'       : 'plan_type_enc',
    'service_type'    : 'service_type_enc',
    'patient_gender'  : 'patient_gender_enc',
    'patient_district': 'patient_district_enc',
}

# ── Malaria season months ──────────────────────────────────────────────────
MALARIA_MONTHS = [4, 5, 10, 11]

# ── Train / Val / Test split ratios ───────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15   # (remaining 15% is test)

# ── Random seed (reproducibility) ─────────────────────────────────────────
RANDOM_STATE = 42

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid')

print('Configuration loaded.')
print(f'Feature columns  : {len(FEATURE_COLS)}')
print(f'Categorical cols : {len(CATEGORICAL_COLS)}')
print(f'Data dir         : {DATA_DIR}')
print(f'Models dir       : {MODELS_DIR}')

---
## Step 2 — Load Data
**Maps to:** `src/data.py → load_raw_data()`

Four CSV tables. `claims` is the central table — we build everything from it.

| Table | Rows | Key columns |
|-------|------|-------------|
| `patients` | 1,500 | patient_id, age, gender, district |
| `members` | 1,500 | member_id, plan_type, enrollment_date |
| `claims` | 2,000 | claim_id, amounts, dates, is_anomaly |
| `payments` | ~1,800 | claim_id, payment_status, arrears |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD DATA  →  src/data.py → load_raw_data(data_dir)
# ═══════════════════════════════════════════════════════════════════════════

patients = pd.read_csv(DATA_DIR + 'zhsf_patients.csv',
                       parse_dates=['date_of_birth', 'registration_date'])
members  = pd.read_csv(DATA_DIR + 'zhsf_members.csv',
                       parse_dates=['enrollment_date', 'expiry_date'])
claims   = pd.read_csv(DATA_DIR + 'zhsf_claims.csv',
                       parse_dates=['service_date', 'submission_date'])
payments = pd.read_csv(DATA_DIR + 'zhsf_payments.csv',
                       parse_dates=['payment_date'])

for name, df in [('patients', patients), ('members', members),
                 ('claims',   claims),   ('payments', payments)]:
    print(f'{name:10s}: {len(df):>5,} rows × {df.shape[1]} columns')

print()
print(f'Claims date range : {claims["service_date"].min().date()}  →  {claims["service_date"].max().date()}')
print(f'Anomaly rate      : {claims["is_anomaly"].mean() * 100:.1f}%  '
      f'({claims["is_anomaly"].sum()} anomalies / {len(claims)} total)')

---
## Step 3 — Data Quality Audit & Cleaning
**Maps to:** `src/data.py → clean_claims(df)`

**Problems found → fixes applied:**
1. Missing amounts → filled with hospital-level median (better than global median)
2. Missing ICD-10 codes → filled with `'UNKNOWN'` placeholder
3. Duplicate `claim_id` rows → dropped, keeping first occurrence
4. Negative lag (service after submission) → clipped to 0

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CLEAN DATA  →  src/data.py → clean_claims(df)
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. Audit missing values ────────────────────────────────────────────────
null_report = (
    claims.isnull().sum()
    .to_frame('null_count')
    .assign(null_pct=lambda x: (x['null_count'] / len(claims) * 100).round(2))
    .query('null_count > 0')
    .sort_values('null_pct', ascending=False)
)
print('=== MISSING VALUES ===')
print(null_report.to_string())

# ── 2. Audit dates and amounts ─────────────────────────────────────────────
lag_check = (claims['submission_date'] - claims['service_date']).dt.days
print('\n=== DATE & AMOUNT CHECKS ===')
print(f'Duplicate claim_id        : {claims.duplicated(["claim_id"]).sum()}')
print(f'Negative lag (bad dates)  : {(lag_check < 0).sum()}')
print(f'Median submission lag     : {lag_check.median():.0f} days')
print(f'Claims submitted > 90 days: {(lag_check > 90).sum()}')
print(f'Min claimed amount (TZS)  : {claims["claimed_amount_tzs"].min():,.0f}')
print(f'Max claimed amount (TZS)  : {claims["claimed_amount_tzs"].max():,.0f}')

# ── 3. Remove duplicates ───────────────────────────────────────────────────
n_before = len(claims)
claims = claims.drop_duplicates(subset=['claim_id'], keep='first').copy()
print(f'\nRemoved {n_before - len(claims)} duplicates → {len(claims):,} rows remain')

# ── 4. Fill missing claimed amounts with per-hospital median ───────────────
# NOTE: we use transform() so the median is broadcast back to each row;
# this is computed on ALL claims here for auditing purposes. The model
# training step re-computes medians on train only (Step 4b below).
hosp_fill = claims.groupby('hospital_id')['claimed_amount_tzs'].transform('median')
claims['claimed_amount_tzs'] = claims['claimed_amount_tzs'].fillna(hosp_fill)

# ── 5. Fill missing ICD-10 codes ──────────────────────────────────────────
claims['icd10_code'] = claims['icd10_code'].fillna('UNKNOWN')

# ── 6. Submission lag (clipped to 0 to handle any bad dates) ──────────────
claims['lag_days'] = (
    (claims['submission_date'] - claims['service_date']).dt.days.clip(lower=0)
)

# ── 7. Verify ──────────────────────────────────────────────────────────────
remaining_nulls = claims[['claimed_amount_tzs', 'icd10_code']].isnull().sum().sum()
print(f'Remaining nulls in key columns: {remaining_nulls}')
print('Data cleaning complete ✓')

---
## Step 4 — Feature Engineering
**Maps to:** `src/features.py → build_features(df, reference_stats=None)`

### ⚠ Leakage-safe design
Features that depend on aggregated statistics (medians, counts) must be computed
from the **training set only**, then looked up for validation/test rows.
Computing them on the full dataset leaks val/test information into features.

This step builds the features on all data for inspection; Step 6 re-computes
reference stats correctly after the split.

| Feature | Fraud signal |
|---------|--------------|
| `claim_age_days` | Very late submissions → possible backdating |
| `is_weekend_submission` | Billing submitted on weekends → unusual |
| `is_malaria_season` | Malaria diagnosis outside season |
| `log_claim_amount` | Log-transform tames extreme outliers |
| `provider_claim_count` | Unusually prolific provider → claim mill |
| `rolling_30d_provider` | Burst of claims in 30-day window → suspicious |
| `cost_deviation_pct` | Charge >> ICD-10 norm → upcoding |
| `amount_vs_hospital_median` | Charge >> facility norm |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING  →  src/features.py → build_temporal_features(df)
# ═══════════════════════════════════════════════════════════════════════════

def build_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add time-based features. No leakage risk — derived from own row only."""
    df = df.copy()
    df['claim_age_days']        = df['lag_days']
    df['submission_month']      = df['submission_date'].dt.month
    df['submission_dayofweek']  = df['submission_date'].dt.dayofweek   # 0=Mon
    df['submission_quarter']    = df['submission_date'].dt.quarter
    df['is_weekend_submission'] = (df['submission_dayofweek'] >= 5).astype(int)
    df['is_malaria_season']     = df['submission_month'].isin(MALARIA_MONTHS).astype(int)
    df['log_claim_amount']      = np.log1p(df['claimed_amount_tzs'])
    return df

claims = build_temporal_features(claims)
print('Temporal features added:', ['claim_age_days', 'submission_month',
      'submission_dayofweek', 'submission_quarter',
      'is_weekend_submission', 'is_malaria_season', 'log_claim_amount'])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FREQUENCY FEATURES  →  src/features.py → build_frequency_features(df, ref)
# ═══════════════════════════════════════════════════════════════════════════

def build_frequency_features(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Total claim counts per provider and hospital.
    Returns (df_with_features, reference_dict) — reference_dict is saved
    for deployment inference on new single claims.
    """
    provider_counts = df.groupby('provider_id')['claim_id'].count().rename('provider_claim_count')
    hospital_counts = df.groupby('hospital_id')['claim_id'].count().rename('hospital_claim_count')

    df = df.merge(provider_counts, on='provider_id', how='left')
    df = df.merge(hospital_counts, on='hospital_id', how='left')

    ref = {'provider_counts': provider_counts, 'hospital_counts': hospital_counts}
    return df, ref


def build_rolling_provider_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIX v2: True 30-day rolling claim count per provider.

    For each claim, counts how many claims the SAME provider submitted
    in the 30 days BEFORE this submission_date (exclusive of current row).
    This is a genuine temporal window — not a cumulative count.

    Requires df to be sorted by submission_date (done inside).
    """
    df = df.sort_values('submission_date').copy()

    rolling_counts = []
    for _, group in df.groupby('provider_id'):
        dates = group['submission_date'].values
        counts = []
        for i, d in enumerate(dates):
            # claims by same provider in the 30 days strictly before d
            window_start = d - np.timedelta64(30, 'D')
            count = int(np.sum((dates[:i] >= window_start) & (dates[:i] < d)))
            counts.append(count)
        rolling_counts.extend(zip(group.index, counts))

    rolling_series = pd.Series(
        dict(rolling_counts), name='rolling_30d_provider'
    )
    df['rolling_30d_provider'] = rolling_series
    return df


claims, _freq_ref_full = build_frequency_features(claims)
claims = build_rolling_provider_feature(claims)

print('Frequency features added.')
print(f"  rolling_30d_provider — mean: {claims['rolling_30d_provider'].mean():.1f}  "
      f"max: {claims['rolling_30d_provider'].max()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# COST-DEVIATION FEATURES  →  src/features.py → build_cost_features(df, ref)
# ═══════════════════════════════════════════════════════════════════════════

def build_cost_features(df: pd.DataFrame,
                        icd10_ref: pd.Series = None,
                        hospital_ref: pd.Series = None
                        ) -> tuple[pd.DataFrame, dict]:
    """
    Compute cost-deviation features.

    If icd10_ref / hospital_ref are provided (deployment / val / test),
    they are used as-is (training-set statistics) to avoid leakage.
    If not provided (full-data exploration run), they are computed from df.
    """
    if icd10_ref is None:
        icd10_ref = df.groupby('icd10_code')['claimed_amount_tzs'].median()
    if hospital_ref is None:
        hospital_ref = df.groupby('hospital_id')['claimed_amount_tzs'].median()

    df = df.copy()
    df['icd10_median_cost']     = df['icd10_code'].map(icd10_ref).fillna(icd10_ref.median())
    df['hospital_median_cost']  = df['hospital_id'].map(hospital_ref).fillna(hospital_ref.median())

    df['cost_deviation_pct'] = (
        (df['claimed_amount_tzs'] - df['icd10_median_cost'])
        / (df['icd10_median_cost'] + 1e-6) * 100
    ).round(2)

    df['amount_vs_hospital_median'] = (
        df['claimed_amount_tzs'] / (df['hospital_median_cost'] + 1e-6)
    ).round(4)

    ref = {'icd10_medians': icd10_ref, 'hospital_medians': hospital_ref}
    return df, ref


claims, _cost_ref_full = build_cost_features(claims)
print('Cost-deviation features added.')
print(f"  cost_deviation_pct — mean: {claims['cost_deviation_pct'].mean():.1f}%  "
      f"p99: {claims['cost_deviation_pct'].quantile(0.99):.0f}%")

---
## Step 5 — Encode Categorical Features
**Maps to:** `src/features.py → encode_categoricals(df, encoders=None)`

One `LabelEncoder` per column, stored in the `ENCODERS` dict and saved to disk.
Deployment code calls `encode_categoricals(df, encoders=loaded_encoders)` to guarantee
the same integer mappings on new data.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ENCODE CATEGORICALS  →  src/features.py → encode_categoricals(df, encoders)
# ═══════════════════════════════════════════════════════════════════════════

def encode_categoricals(df: pd.DataFrame,
                        encoders: dict = None,
                        fit: bool = True
                        ) -> tuple[pd.DataFrame, dict]:
    """
    Encode categorical columns using LabelEncoder.

    Parameters
    ----------
    df       : input DataFrame
    encoders : pre-fitted encoder dict (None → fit new encoders)
    fit      : if True, fit new encoders on df; if False, use provided encoders

    Returns
    -------
    (df_encoded, encoders_dict)
    """
    df = df.copy()
    if encoders is None:
        encoders = {}

    for raw_col, enc_col in CATEGORICAL_COLS.items():
        if fit or raw_col not in encoders:
            le = LabelEncoder()
            df[enc_col] = le.fit_transform(df[raw_col].astype(str))
            encoders[raw_col] = le
        else:
            le = encoders[raw_col]
            values = df[raw_col].astype(str)
            # Map unseen categories to a fallback (0)
            df[enc_col] = values.apply(
                lambda v: int(le.transform([v])[0]) if v in le.classes_ else 0
            )

    return df, encoders


claims, ENCODERS = encode_categoricals(claims, fit=True)

print('Categorical encoding complete.')
for col, le in ENCODERS.items():
    print(f'  {col:20s} → classes: {list(le.classes_)}')

In [ ]:
# ── Feature summary statistics ─────────────────────────────────────────────
print(f'Total features in FEATURE_COLS: {len(FEATURE_COLS)}')
print()

# Verify every feature column exists and has no nulls
missing_cols = [c for c in FEATURE_COLS if c not in claims.columns]
if missing_cols:
    raise ValueError(f'Missing feature columns: {missing_cols}')

null_in_features = claims[FEATURE_COLS].isnull().sum()
if null_in_features.any():
    print('⚠ Nulls in features:')
    print(null_in_features[null_in_features > 0])
else:
    print('✓ All feature columns present and null-free')

claims[FEATURE_COLS].describe().T.round(2)

---
## Step 6 — Chronological Split + Leakage-Safe Reference Stats
**Maps to:** `src/data.py → chronological_split(df)` + `src/features.py → recompute_refs(train_df)`

```
──────────────────────────────────────────────────────► time
│──────── 70% TRAIN ────────│── 15% VAL ──│── 15% TEST ──│
```

**Leakage fix:** After splitting, reference statistics (ICD-10 medians, hospital medians,
provider counts) are re-computed **from the training set only**. Val and test rows then
look up their features from this training reference — exactly as production would do.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SPLIT  →  src/data.py → chronological_split(df)
# ═══════════════════════════════════════════════════════════════════════════

claims = claims.sort_values('service_date').reset_index(drop=True)

n         = len(claims)
train_end = int(n * TRAIN_RATIO)
val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df_raw = claims.iloc[:train_end].copy()
val_df_raw   = claims.iloc[train_end:val_end].copy()
test_df_raw  = claims.iloc[val_end:].copy()

for name, df in [('Train', train_df_raw), ('Validation', val_df_raw), ('Test', test_df_raw)]:
    anom = df['is_anomaly'].mean() * 100
    print(f'{name:12s}: {len(df):>5,} rows | '
          f'{df["service_date"].min().date()} → {df["service_date"].max().date()} | '
          f'anomaly rate: {anom:.1f}% ({df["is_anomaly"].sum()} anomalies)')

print()

# ═══════════════════════════════════════════════════════════════════════════
# LEAKAGE-SAFE: Compute reference stats from TRAIN only
# Then apply to val/test via lookup (same as production would do)
# ═══════════════════════════════════════════════════════════════════════════

# Frequency references from train only
TRAIN_PROVIDER_COUNTS  = train_df_raw.groupby('provider_id')['claim_id'].count()
TRAIN_HOSPITAL_COUNTS  = train_df_raw.groupby('hospital_id')['claim_id'].count()

# Cost references from train only
TRAIN_ICD10_MEDIANS    = train_df_raw.groupby('icd10_code')['claimed_amount_tzs'].median()
TRAIN_HOSPITAL_MEDIANS = train_df_raw.groupby('hospital_id')['claimed_amount_tzs'].median()


def apply_train_refs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply training-set reference statistics to any split.
    Unknown providers/hospitals/ICD-10s fall back to the training median.
    This is exactly what the deployment inference function must also do.
    """
    df = df.copy()

    # Frequency lookups
    df['provider_claim_count'] = (
        df['provider_id'].map(TRAIN_PROVIDER_COUNTS).fillna(1).astype(int)
    )
    df['hospital_claim_count'] = (
        df['hospital_id'].map(TRAIN_HOSPITAL_COUNTS).fillna(1).astype(int)
    )

    # Cost lookups
    icd_fallback  = float(TRAIN_ICD10_MEDIANS.median())
    hosp_fallback = float(TRAIN_HOSPITAL_MEDIANS.median())

    df['icd10_median_cost']    = df['icd10_code'].map(TRAIN_ICD10_MEDIANS).fillna(icd_fallback)
    df['hospital_median_cost'] = df['hospital_id'].map(TRAIN_HOSPITAL_MEDIANS).fillna(hosp_fallback)

    df['cost_deviation_pct'] = (
        (df['claimed_amount_tzs'] - df['icd10_median_cost'])
        / (df['icd10_median_cost'] + 1e-6) * 100
    ).round(2)

    df['amount_vs_hospital_median'] = (
        df['claimed_amount_tzs'] / (df['hospital_median_cost'] + 1e-6)
    ).round(4)

    return df


train_df = apply_train_refs(train_df_raw)
val_df   = apply_train_refs(val_df_raw)
test_df  = apply_train_refs(test_df_raw)

# Recompute rolling_30d_provider on each split independently
train_df = build_rolling_provider_feature(train_df)
val_df   = build_rolling_provider_feature(val_df)
test_df  = build_rolling_provider_feature(test_df)

# Extract X/y arrays
X_train = train_df[FEATURE_COLS].values
y_train = train_df['is_anomaly'].values

X_val   = val_df[FEATURE_COLS].values
y_val   = val_df['is_anomaly'].values

X_test  = test_df[FEATURE_COLS].values
y_test  = test_df['is_anomaly'].values

print(f'X_train shape: {X_train.shape}  |  anomalies: {y_train.sum()}')
print(f'X_val   shape: {X_val.shape}  |  anomalies: {y_val.sum()}')
print(f'X_test  shape: {X_test.shape}  |  anomalies: {y_test.sum()}')

---
## Step 7 — Balance Classes with SMOTE
**Maps to:** `src/train.py → smote_balance(X, y)`

With only ~4% anomalies the training set is heavily skewed.
SMOTE generates synthetic anomaly rows so the model sees a 50/50 split.

> SMOTE is applied **only to the training set**. Val and test stay untouched.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SMOTE  →  src/train.py → smote_balance(X, y, random_state)
# ═══════════════════════════════════════════════════════════════════════════

print(f'Before SMOTE — Normal: {(y_train==0).sum():,}  Anomaly: {(y_train==1).sum():,}')

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After  SMOTE — Normal: {(y_train_sm==0).sum():,}  Anomaly: {(y_train_sm==1).sum():,}')
print(f'\nData is now balanced (50/50) for training.')

---
## Step 8 — Train XGBoost (Primary Model)
**Maps to:** `src/train.py → train_xgboost(X_train, y_train, X_val, y_val)`

### ⚠ Bug fix: `scale_pos_weight` removed

In v1, `scale_pos_weight = 23` was set **on top of** SMOTE-balanced data.
These two techniques both address class imbalance — using both simultaneously
double-penalizes the majority class and produces uncalibrated probabilities
that are inflated toward 1, causing the F1-optimizer to select an
operationally meaningless threshold of ~0.09.

**Rule:** Use **one** of: SMOTE **OR** `scale_pos_weight`, not both.

Since we are using SMOTE, `scale_pos_weight=1` (the default, neutral value).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAIN XGBOOST  →  src/train.py → train_xgboost(...)
# ═══════════════════════════════════════════════════════════════════════════

# FIX v2: scale_pos_weight=1 because SMOTE already balanced the classes.
# Using scale_pos_weight=23 + SMOTE simultaneously inflates anomaly scores,
# which collapses the F1-optimal threshold to ~0.09 (operationally wrong).

xgb_model = XGBClassifier(
    n_estimators          = 300,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    scale_pos_weight      = 1,     # FIX: neutral — SMOTE already balanced training data
    eval_metric           = 'logloss',
    early_stopping_rounds = 20,
    random_state          = RANDOM_STATE,
    n_jobs                = -1
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print(f'\nBest number of trees: {xgb_model.best_iteration}')

---
## Step 9 — Train Random Forest (A/B Challenger Model)
**Maps to:** `src/train.py → train_random_forest(X_train, y_train)`

Random Forest builds trees in parallel on bootstrap samples.
Because training data is already balanced by SMOTE, `class_weight='balanced'`
is set to `None` — no double-weighting.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAIN RANDOM FOREST  →  src/train.py → train_random_forest(...)
# ═══════════════════════════════════════════════════════════════════════════

rf_model = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = 10,
    min_samples_leaf = 3,
    class_weight     = None,   # SMOTE already balanced — no additional weighting
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)

rf_model.fit(X_train_sm, y_train_sm)
print('Random Forest training complete.')

---
## Step 10 — Threshold Selection (Validation Set)
**Maps to:** `src/evaluate.py → find_f1_threshold(y_true, y_proba)`

### Why this matters
Both models output a **probability** (0→1). We flag a claim when the score exceeds a threshold.

| Threshold | Effect |
|-----------|--------|
| Too low (e.g. 0.02) | Nearly every claim flagged → auditors overwhelmed, precision ≈ 4% |
| Too high (e.g. 0.95) | Almost nothing flagged → real fraud missed |
| **F1-optimal** | Best precision/recall balance for operational use |

Threshold is selected on **validation set only** to avoid data leakage into the test evaluation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SCORE VALIDATION SET  →  src/evaluate.py
# ═══════════════════════════════════════════════════════════════════════════

xgb_proba_val = xgb_model.predict_proba(X_val)[:, 1]
rf_proba_val  = rf_model.predict_proba(X_val)[:, 1]

print('Validation set probability distributions:')
print(f'  XGB — min: {xgb_proba_val.min():.4f}  '
      f'mean: {xgb_proba_val.mean():.4f}  '
      f'max: {xgb_proba_val.max():.4f}')
print(f'  RF  — min: {rf_proba_val.min():.4f}  '
      f'mean: {rf_proba_val.mean():.4f}  '
      f'max: {rf_proba_val.max():.4f}')
print()
print('With the double-weighting bug fixed, XGB scores should now be')
print('better calibrated and the optimal threshold should be > 0.3')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIND OPTIMAL THRESHOLD  →  src/evaluate.py → find_f1_threshold(...)
# ═══════════════════════════════════════════════════════════════════════════

def find_f1_threshold(y_true: np.ndarray,
                      y_proba: np.ndarray,
                      model_name: str = 'Model'
                      ) -> tuple[float, np.ndarray, np.ndarray]:
    """
    Sweep thresholds 0.01 → 0.99 and return the one that maximises F1.

    Returns
    -------
    (best_threshold, threshold_array, f1_score_array)
    """
    thresholds = np.linspace(0.01, 0.99, 200)
    f1_scores  = [
        f1_score(y_true, (y_proba >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_idx = int(np.argmax(f1_scores))
    best_thr = float(thresholds[best_idx])
    best_f1  = float(f1_scores[best_idx])
    print(f'{model_name:15s} — F1-optimal threshold: {best_thr:.3f}  (F1 = {best_f1:.3f})')
    return best_thr, thresholds, np.array(f1_scores)


xgb_threshold, xgb_thr_range, xgb_f1_curve = find_f1_threshold(
    y_val, xgb_proba_val, 'XGBoost')

rf_threshold, rf_thr_range, rf_f1_curve = find_f1_threshold(
    y_val, rf_proba_val, 'RandomForest')

# ── Plot threshold vs F1 curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, thr_r, f1_c, opt, label, color in [
    (axes[0], xgb_thr_range, xgb_f1_curve, xgb_threshold, 'XGBoost',      'steelblue'),
    (axes[1], rf_thr_range,  rf_f1_curve,  rf_threshold,  'Random Forest', 'darkorange'),
]:
    ax.plot(thr_r, f1_c, color=color, linewidth=2)
    ax.axvline(opt, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Optimal = {opt:.3f}')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('F1-Score (Validation)')
    ax.set_title(f'{label} — Threshold vs F1', fontweight='bold')
    ax.legend()
    ax.set_xlim([0, 1])

plt.suptitle('Threshold Selection on Validation Set (not test set)', y=1.02)
plt.tight_layout()
plt.savefig('../figures/threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PRECISION-RECALL CURVES  →  src/evaluate.py → plot_pr_curves(...)
# ═══════════════════════════════════════════════════════════════════════════

xgb_ap = average_precision_score(y_val, xgb_proba_val)
rf_ap  = average_precision_score(y_val, rf_proba_val)

fig, ax = plt.subplots(figsize=(8, 5))

for proba, ap, label, color, thr in [
    (xgb_proba_val, xgb_ap, 'XGBoost',       'steelblue',   xgb_threshold),
    (rf_proba_val,  rf_ap,  'Random Forest',  'darkorange',  rf_threshold),
]:
    prec, rec, _ = precision_recall_curve(y_val, proba)
    ax.plot(rec, prec, color=color, linewidth=2,
            label=f'{label}  (AP={ap:.3f}, thr={thr:.3f})')

ax.axhline(0.85, color='grey', linestyle='--', linewidth=1.2,
           label='85% precision target')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Validation Set', fontweight='bold')
ax.legend(loc='lower left')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('../figures/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'XGBoost   AP: {xgb_ap:.3f}')
print(f'RandomFor AP: {rf_ap:.3f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VALIDATION METRICS  →  src/evaluate.py → full_report(y_true, y_pred, y_proba)
# ═══════════════════════════════════════════════════════════════════════════

def full_report(y_true: np.ndarray,
                y_pred: np.ndarray,
                y_proba: np.ndarray,
                label: str,
                threshold: float) -> dict:
    """Print + return a comprehensive metrics dict."""
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_proba)
    ap   = average_precision_score(y_true, y_proba)
    cm   = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f'\n══════ {label} (threshold={threshold:.3f}) ══════')
    print(f'  Precision  : {prec*100:>5.1f}%')
    print(f'  Recall     : {rec*100:>5.1f}%')
    print(f'  F1-Score   : {f1*100:>5.1f}%')
    print(f'  AUC-ROC    : {auc:.4f}')
    print(f'  Avg Prec   : {ap:.4f}')
    print(f'  Flags      : {y_pred.sum()} / {len(y_pred)} claims  '
          f'({y_pred.mean()*100:.1f}% flag rate)')
    print(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp}')

    return dict(label=label, precision=prec, recall=rec, f1=f1,
                auc_roc=auc, avg_prec=ap, threshold=threshold,
                tn=tn, fp=fp, fn=fn, tp=tp, n_flags=int(y_pred.sum()))


xgb_pred_val = (xgb_proba_val >= xgb_threshold).astype(int)
rf_pred_val  = (rf_proba_val  >= rf_threshold).astype(int)

val_xgb_metrics = full_report(y_val, xgb_pred_val, xgb_proba_val,
                               'XGBoost (Validation)', xgb_threshold)
val_rf_metrics  = full_report(y_val, rf_pred_val,  rf_proba_val,
                               'Random Forest (Validation)', rf_threshold)

---
## Step 10b — Feature Importance
**Maps to:** `src/evaluate.py → plot_feature_importance(model, feature_cols)`

Understanding which features drive each model helps validate that the model
has learned genuine fraud signals — not noise or data artefacts.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FEATURE IMPORTANCE  →  src/evaluate.py → plot_feature_importance(...)
# ═══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# XGBoost — gain-based importance
xgb_imp = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

xgb_imp.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('XGBoost — Feature Importance (gain)', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Random Forest — mean decrease in impurity
rf_imp = pd.Series(
    rf_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

rf_imp.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Random Forest — Feature Importance (MDI)', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features — XGBoost:')
print(xgb_imp.tail(5).sort_values(ascending=False).to_string())
print()
print('Top 5 features — Random Forest:')
print(rf_imp.tail(5).sort_values(ascending=False).to_string())

---
## Step 11 — Final Evaluation on the Held-Out Test Set
**Maps to:** `src/evaluate.py → full_report(...)` on test split

The test set has been **locked** throughout — it was never used for threshold selection
or any other decision. This gives an unbiased estimate of real-world performance.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FINAL TEST EVALUATION  →  src/evaluate.py
# ═══════════════════════════════════════════════════════════════════════════

xgb_proba_test = xgb_model.predict_proba(X_test)[:, 1]
rf_proba_test  = rf_model.predict_proba(X_test)[:, 1]

xgb_pred_test  = (xgb_proba_test >= xgb_threshold).astype(int)
rf_pred_test   = (rf_proba_test  >= rf_threshold).astype(int)

test_xgb_metrics = full_report(y_test, xgb_pred_test, xgb_proba_test,
                                'XGBoost (TEST — FINAL)', xgb_threshold)
test_rf_metrics  = full_report(y_test, rf_pred_test,  rf_proba_test,
                                'Random Forest (TEST — FINAL)', rf_threshold)

# ── Confusion matrices ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pred, title in [
    (axes[0], xgb_pred_test, f'XGBoost (thr={xgb_threshold:.3f})'),
    (axes[1], rf_pred_test,  f'Random Forest (thr={rf_threshold:.3f})'),
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: Normal', 'Pred: Anomaly'],
                yticklabels=['Act: Normal',  'Act: Anomaly'],
                annot_kws={'size': 13})
    ax.set_title(f'{title} — Test Set', fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Side-by-side comparison table ─────────────────────────────────────────
comparison = pd.DataFrame([
    {k: v for k, v in test_xgb_metrics.items() if k != 'label'},
    {k: v for k, v in test_rf_metrics.items()  if k != 'label'},
], index=['XGBoost', 'Random Forest'])

print('\n=== TEST SET COMPARISON TABLE ===')
print(comparison[['precision','recall','f1','auc_roc','avg_prec',
                   'threshold','n_flags','tp','fp','fn']].to_string())

---
## Step 12 — Save All Deployment Artifacts
**Maps to:** `src/artifacts.py → save_all(models_dir, ...)`

| Artifact | File | Needed by |
|----------|------|----------|
| XGBoost | `xgb_anomaly_detector.pkl` | Scoring engine |
| Random Forest | `rf_anomaly_detector.pkl` | A/B challenger |
| Feature list | `feature_cols.pkl` | Column order validation |
| Encoders | `encoders.pkl` | Categorical mapping |
| Thresholds | `thresholds.pkl` | Binary flag decision |
| ICD-10 medians | `icd10_medians.pkl` | `cost_deviation_pct` feature |
| Hospital medians | `hospital_medians.pkl` | `amount_vs_hospital_median` feature |
| Provider counts | `provider_counts.pkl` | `provider_claim_count` feature |
| Hospital counts | `hospital_counts.pkl` | `hospital_claim_count` feature |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SAVE ARTIFACTS  →  src/artifacts.py → save_all(models_dir, ...)
# ═══════════════════════════════════════════════════════════════════════════

os.makedirs('../figures', exist_ok=True)

# Models
joblib.dump(xgb_model,    MODELS_DIR + 'xgb_anomaly_detector.pkl')
joblib.dump(rf_model,     MODELS_DIR + 'rf_anomaly_detector.pkl')

# Feature schema
joblib.dump(FEATURE_COLS, MODELS_DIR + 'feature_cols.pkl')

# Encoders (one per categorical column)
joblib.dump(ENCODERS,     MODELS_DIR + 'encoders.pkl')

# Decision thresholds (F1-optimal on validation set)
THRESHOLDS = {'xgboost': xgb_threshold, 'random_forest': rf_threshold}
joblib.dump(THRESHOLDS,   MODELS_DIR + 'thresholds.pkl')

# Reference statistics (train-set only — no leakage)
joblib.dump(TRAIN_ICD10_MEDIANS,    MODELS_DIR + 'icd10_medians.pkl')
joblib.dump(TRAIN_HOSPITAL_MEDIANS, MODELS_DIR + 'hospital_medians.pkl')
joblib.dump(TRAIN_PROVIDER_COUNTS,  MODELS_DIR + 'provider_counts.pkl')
joblib.dump(TRAIN_HOSPITAL_COUNTS,  MODELS_DIR + 'hospital_counts.pkl')

print(f'=== ARTIFACTS SAVED TO {MODELS_DIR} ===')
for f in sorted(os.listdir(MODELS_DIR)):
    size = os.path.getsize(f'{MODELS_DIR}{f}')
    print(f'  {f:<35s}  {size/1024:>7.1f} KB')

print()
print('=== DEPLOYMENT THRESHOLDS ===')
for model, thr in THRESHOLDS.items():
    print(f'  {model:<20s}: {thr:.4f}')

---
## Step 13 — Smoke Test: Predict 2 Sample Claims
**Maps to:** `src/predict.py → prepare_sample()` + `predict_claim()`

Simulates exactly what the production API endpoint does:
1. Accept raw claim dict
2. Load saved artifacts
3. Apply feature engineering
4. Apply saved encoders
5. Score with both models
6. Apply per-model threshold → binary flag

**Sample A** — routine claim → expected ✅ Normal  
**Sample B** — suspicious claim (89-day lag, 16× hospital median) → expected 🚨 Anomaly

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD ARTIFACTS (fresh, as a production process would)  →  src/artifacts.py
# ═══════════════════════════════════════════════════════════════════════════

_xgb_model     = joblib.load(MODELS_DIR + 'xgb_anomaly_detector.pkl')
_rf_model      = joblib.load(MODELS_DIR + 'rf_anomaly_detector.pkl')
_feature_cols  = joblib.load(MODELS_DIR + 'feature_cols.pkl')
_encoders      = joblib.load(MODELS_DIR + 'encoders.pkl')
_thresholds    = joblib.load(MODELS_DIR + 'thresholds.pkl')
_icd10_ref     = joblib.load(MODELS_DIR + 'icd10_medians.pkl')
_hospital_ref  = joblib.load(MODELS_DIR + 'hospital_medians.pkl')
_provider_ref  = joblib.load(MODELS_DIR + 'provider_counts.pkl')
_hospital_cnt  = joblib.load(MODELS_DIR + 'hospital_counts.pkl')

print('All artifacts loaded.')
print(f'XGBoost threshold    : {_thresholds["xgboost"]:.4f}')
print(f'RandomForest threshold: {_thresholds["random_forest"]:.4f}')
print(f'Feature columns      : {len(_feature_cols)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTIONS  →  src/predict.py
# ═══════════════════════════════════════════════════════════════════════════

def prepare_sample(raw: dict) -> pd.DataFrame:
    """
    Convert a raw claim dict into a model-ready feature row.

    Expected keys
    -------------
    service_date, submission_date, claimed_amount_tzs,
    facility_type, plan_type, service_type,
    patient_gender, patient_district,
    icd10_code, hospital_id, provider_id

    Returns
    -------
    pd.DataFrame with columns exactly matching _feature_cols
    """
    r      = raw.copy()
    svc    = pd.Timestamp(r['service_date'])
    sub    = pd.Timestamp(r['submission_date'])
    lag    = max(0, (sub - svc).days)
    amount = float(r['claimed_amount_tzs'])

    # ── Temporal ───────────────────────────────────────────────────────────
    row = {
        'claim_age_days'       : lag,
        'submission_month'     : sub.month,
        'submission_dayofweek' : sub.dayofweek,
        'submission_quarter'   : sub.quarter,
        'is_weekend_submission': int(sub.dayofweek >= 5),
        'is_malaria_season'    : int(sub.month in MALARIA_MONTHS),
        'log_claim_amount'     : np.log1p(amount),
    }

    # ── Frequency lookups ──────────────────────────────────────────────────
    # FIX v2: provider and hospital use their own correct reference dict
    row['provider_claim_count'] = int(_provider_ref.get(r['provider_id'], 1))
    row['hospital_claim_count'] = int(_hospital_cnt.get(r['hospital_id'], 1))  # FIX: was _provider_ref
    row['rolling_30d_provider'] = 0   # unknown for single new claim

    # ── Cost-deviation lookups ─────────────────────────────────────────────
    icd      = str(r.get('icd10_code', 'UNKNOWN') or 'UNKNOWN')
    icd_med  = float(_icd10_ref.get(icd,          _icd10_ref.median()))
    hosp_med = float(_hospital_ref.get(r['hospital_id'], _hospital_ref.median()))

    row['icd10_median_cost']         = icd_med
    row['cost_deviation_pct']        = round((amount - icd_med) / (icd_med + 1e-6) * 100, 2)
    row['amount_vs_hospital_median'] = round(amount / (hosp_med + 1e-6), 4)

    # ── Categorical encoding ───────────────────────────────────────────────
    for raw_col, enc_col in CATEGORICAL_COLS.items():
        le  = _encoders[raw_col]
        val = str(r.get(raw_col, ''))
        row[enc_col] = int(le.transform([val])[0]) if val in le.classes_ else 0

    return pd.DataFrame([row])[_feature_cols]


def predict_claim(raw: dict, label: str = 'Claim') -> dict:
    """Score a raw claim dict with both models and return a result dict."""
    X = prepare_sample(raw).values

    xgb_score = float(_xgb_model.predict_proba(X)[0, 1])
    rf_score  = float(_rf_model.predict_proba(X)[0, 1])
    xgb_flag  = xgb_score >= _thresholds['xgboost']
    rf_flag   = rf_score  >= _thresholds['random_forest']

    print(f'\n═══ {label} ═══')
    print(f'  XGBoost      score: {xgb_score:.4f}  threshold: {_thresholds["xgboost"]:.4f}  '
          f'→ {"🚨 ANOMALY" if xgb_flag else "✅ Normal"}')
    print(f'  RandomForest score: {rf_score:.4f}  threshold: {_thresholds["random_forest"]:.4f}  '
          f'→ {"🚨 ANOMALY" if rf_flag else "✅ Normal"}')

    return dict(xgb_score=xgb_score, rf_score=rf_score,
                xgb_flag=xgb_flag,   rf_flag=rf_flag)


print('Inference functions defined.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SAMPLE PREDICTIONS  →  test/sample_claims.py
# ═══════════════════════════════════════════════════════════════════════════

# ── Sample A: Routine outpatient — expected ✅ Normal ──────────────────────
sample_A = {
    'service_date'       : '2023-11-05',
    'submission_date'    : '2023-11-12',   # 7-day lag — normal
    'claimed_amount_tzs' : 35_000,         # typical outpatient amount
    'facility_type'      : 'Public',
    'plan_type'          : 'Standard',
    'service_type'       : 'Outpatient',
    'patient_gender'     : 'Female',
    'patient_district'   : 'Urban West',
    'icd10_code'         : 'I10',           # Essential hypertension
    'hospital_id'        : 'HOS011',
    'provider_id'        : 'PRV0036',
}

# ── Sample B: Suspicious claim — expected 🚨 Anomaly ──────────────────────
# Three overlapping red flags:
#   (1) 89-day submission lag — far above normal median of 11 days
#   (2) Amount 950,000 TZS — ~16× the hospital median
#   (3) Malaria diagnosis in August — out of malaria season (Apr, May, Oct, Nov)
sample_B = {
    'service_date'       : '2023-06-01',
    'submission_date'    : '2023-08-29',   # 89-day lag — suspicious
    'claimed_amount_tzs' : 950_000,        # ~16× hospital median
    'facility_type'      : 'Private',
    'plan_type'          : 'Family',
    'service_type'       : 'Inpatient',
    'patient_gender'     : 'Male',
    'patient_district'   : 'Chake-Chake',
    'icd10_code'         : 'B54',           # Unspecified malaria — out of season
    'hospital_id'        : 'HOS008',
    'provider_id'        : 'PRV0118',
}

result_A = predict_claim(sample_A, 'Sample A — Routine Claim (expected: Normal)')
result_B = predict_claim(sample_B, 'Sample B — Suspicious Claim (expected: Anomaly)')

---
## Summary

| Step | What We Did | Key Output | `src/` mapping |
|------|-------------|------------|----------------|
| 1 | Config & imports | Paths, constants, feature list | `config.py` |
| 2 | Load data | 4 ZHSF tables | `data.py → load_raw_data()` |
| 3 | Clean data | Filled nulls, removed duplicates | `data.py → clean_claims()` |
| 4 | Feature engineering | 18 fraud-signal features | `features.py → build_features()` |
| 5 | Encode categoricals | Per-column LabelEncoders | `features.py → encode_categoricals()` |
| 6 | Chronological split | Train 70% / Val 15% / Test 15% | `data.py → chronological_split()` |
| 6b | Leakage-safe refs | Stats from train only | `features.py → apply_train_refs()` |
| 7 | SMOTE | Balanced training classes | `train.py → smote_balance()` |
| 8 | XGBoost | Primary model, `scale_pos_weight=1` | `train.py → train_xgboost()` |
| 9 | Random Forest | A/B challenger | `train.py → train_random_forest()` |
| 10 | Threshold selection | F1-optimal on val set | `evaluate.py → find_f1_threshold()` |
| 10b | Feature importance | Top fraud drivers | `evaluate.py → plot_feature_importance()` |
| 11 | Test evaluation | Unbiased final metrics | `evaluate.py → full_report()` |
| 12 | Save artifacts | 9 pkl files | `artifacts.py → save_all()` |
| 13 | Smoke test | 2 sample claims | `predict.py → predict_claim()` |

### Bugs fixed in v2
1. `scale_pos_weight=23` + SMOTE → removed double-penalty; threshold now realistic (0.3–0.6 range)
2. `hospital_claim_count` lookup used `_provider_ref` → fixed to use `_hospital_cnt`
3. Reference stats computed on full dataset → now computed on train split only
4. `rolling_30d_provider` was a cumcount → now a true 30-day window

### Deployment artifact checklist
```
models/
├── xgb_anomaly_detector.pkl    # Primary model
├── rf_anomaly_detector.pkl     # A/B challenger
├── feature_cols.pkl            # Column order
├── encoders.pkl                # Categorical mappings
├── thresholds.pkl              # Per-model F1-optimal thresholds
├── icd10_medians.pkl           # Reference: cost deviation
├── hospital_medians.pkl        # Reference: cost deviation
├── provider_counts.pkl         # Reference: frequency features
└── hospital_counts.pkl         # Reference: frequency features
```